# Liquidity Management
**Luxembourg Regulatory Framework — UCITS / AIFMD**

1. **Liquidity profile** — days-to-liquidate, bucket distribution, redemption stress, investor concentration, liquidity-adjusted VaR
2. **LMT trigger analysis (MRS-84)** — 12-month gate / swing / suspension simulation per AIFMD II Art. 16a

**PE Buyout** and **Infra Core** are closed-ended with no redemption obligation; they appear in the profile section only.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.database      import get_engine
from src.risk_utils    import (
    days_to_liquidate, liquidity_buckets,
    redemption_stress, investor_concentration,
    liquidity_adjusted_var, lmt_trigger_analysis,
)
from src.plot_style import (
    C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
    apply_ax_style, section_title, fig_header, fig_footer,
    callout_box, threshold_vline, threshold_hline, breach_fill,
    pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
)
from src.nb_utils import save_fig, save_table, styled_table, save_table_png


ENGINE  = get_engine()
QUARTER = '2026-03-31'

## Liquids

### 1. Liquidity Profile

Days-to-liquidate and ESMA bucket distribution across all six funds. Positions loaded from the database at the standard valuation date. PE Buyout and Infra Core are shown for completeness but hold no liquid market positions. We wil linclude this exercise here only for HF and UCITS


In [ ]:
VALUATION_DATE = '2026-05-13'
BUCKET_ORDER   = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-365 days', '> 1 year']

OPEN_FUNDS = {
    'UCITS_Balanced':   'UCITS Balanced',
    'AIFM_HedgeFund':   'AIFM Hedge Fund',
}
ALL_FUNDS  = {**OPEN_FUNDS}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes = axes.flatten()

for ax, (fid, fname) in zip(axes, ALL_FUNDS.items()):

    pos = pd.read_sql(
        "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
        ENGINE, params={'fid': fid, 'dt': VALUATION_DATE}
    )
    nav  = pos['market_value_eur'].sum()
    liq  = liquidity_buckets(days_to_liquidate(pos, pct_adv=0.25))
    bkt  = (liq.groupby('liquidity_bucket', observed=True)['market_value_eur']
               .sum().reindex(BUCKET_ORDER, fill_value=0.0))
    pcts = (bkt / nav * 100).values

    bars = ax.bar(range(6), pcts, color=C['cyan'], width=0.6)
    ax.set_xticks(range(6))
    ax.set_xticklabels(['1d', '2-7d', '8-30d', '31-90d', '91-365d', '>1yr'], fontsize=8)
    ax.set_ylabel('% of NAV', fontsize=8)
    section_title(ax, fname)
    ax.set_ylim(0, 115)
    for bar, pct in zip(bars, pcts):
        if pct > 2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{pct:.0f}%', ha='center', va='bottom', fontsize=7)

fig.suptitle('ESMA Liquidity Bucket Distribution — All Funds', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


#### 1.2 Redemption Stress — Single-Period Snapshot

Point-in-time gate analysis: can each open-ended fund meet a 25% redemption within a 5-day notice period? PE Buyout and Infra Core are excluded — no redemption obligation.


In [ ]:
stress_rows = []

for fid, fname in OPEN_FUNDS.items():
    pos = pd.read_sql(
        "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
        ENGINE, params={'fid': fid, 'dt': VALUATION_DATE}
    )
    nav = pos['market_value_eur'].sum()
    liq = liquidity_buckets(days_to_liquidate(pos, pct_adv=0.25))
    rs  = redemption_stress(liq, nav, redemption_pct=0.25, notice_days=5)
    stress_rows.append({
        'Fund'              : fname,
        'NAV (€m)'          : round(nav / 1e6, 1),
        'Redemption (€m)'   : round(rs['redemption_amount_eur'] / 1e6, 1),
        'Liquid Assets (€m)': round(rs['liquid_assets_eur'] / 1e6, 1),
        'Coverage Ratio'    : rs['coverage_ratio'],
        'Can Meet'          : '✓' if rs['can_meet_redemption'] else '✗',
        'Recommendation'    : rs['recommendation'],
    })

pd.DataFrame(stress_rows).set_index('Fund')


### 2 Investor Base Model -- Modeling the Redemption Schedule

Rather than specifying a redemption schedule as a fixed list of numbers, we derive it from first principles using a per-fund **investor registry**. Each fund's investor base is decomposed into investor types, each characterised by:

| Notation | Meaning |
|---|---|
| $w_i$ | AUM share of investor type $i$ ($\sum_i w_i = 1$) |
| $\mu_i^{\text{norm}}$ | Monthly redemption rate under normal conditions |
| $\mu_i^{\text{stress}}$ | Monthly redemption rate under stress conditions |
| $\sigma$ | Lognormal dispersion parameter (per fund) |

#### Normal months -- lognormal draws

For months not designated as stress months, each investor type draws from a lognormal distribution centred on its normal rate. The lognormal is chosen because redemption rates are strictly positive and right-skewed -- occasional large outflows occur with low but non-negligible probability:

$$X_{i,t} \sim \text{LogNormal}\!\left(\ln\mu_i^{\text{norm}} - \tfrac{\sigma^2}{2},\;\; \sigma\right)$$

Parametrised this way, $\mathbb{E}[X_{i,t}] = \mu_i^{\text{norm}}$ exactly. The dispersion $\sigma$ is calibrated higher for hedge fund investor bases, which exhibit more volatile redemption behaviour than retail or balanced fund investors.

The aggregate monthly redemption rate is the AUM-weighted sum across all investor types:

$$r_t = \sum_{i=1}^{N} w_i \cdot X_{i,t}$$

#### Stress months -- deterministic override

Designated stress months $t \in \mathcal{T}_{\text{stress}}$ override the lognormal draw and use each investor type's stress rate directly:

$$r_t = \sum_{i=1}^{N} w_i \cdot \mu_i^{\text{stress}} \quad \forall\; t \in \mathcal{T}_{\text{stress}}$$

This models a coordinated redemption wave -- e.g. following a market shock, prime broker margin call, or manager headline risk -- where all investor types simultaneously redeem at elevated rates. Stress months are parameterised per fund.

#### Weighted reference rates

The two reference lines on the schedule chart are:

$$r^{\text{norm,wt}} = \sum_i w_i \cdot \mu_i^{\text{norm}} \qquad r^{\text{stress,wt}} = \sum_i w_i \cdot \mu_i^{\text{stress}}$$

These are the expected aggregate rates under normal and full-stress conditions respectively.

In [ ]:
INVESTOR_REGISTRY = {
    'UCITS_Balanced': {
        'label'        : 'UCITS Balanced',
        'investors'    : [
            #  type              w      normal  stress
            ('Retail',          0.40,  0.030,  0.120),
            ('Institutional',   0.45,  0.040,  0.180),
            ('Family Office',   0.15,  0.020,  0.100),
        ],
        'stress_months': {3, 4},
        'sigma'        : 0.30,    # moderate -- mixed retail/institutional base
        'seed'         : 42,
    },
    'AIFM_HedgeFund': {
        'label'        : 'AIFM Hedge Fund',
        'investors'    : [
            ('Pension Fund',    0.50,  0.030,  0.200),
            ('Fund of Funds',   0.30,  0.050,  0.250),
            ('Family Office',   0.20,  0.040,  0.180),
        ],
        'stress_months': {3, 4},
        'sigma'        : 0.40,    # higher -- institutional base reacts faster and larger
        'seed'         : 42,
    },
}


def build_redemption_schedule(reg, n_months=12):
    """
    Generate a monthly redemption schedule from investor base parameters.

    Normal months: AUM-weighted lognormal draw per investor type,
    E[draw] = normal_rate by lognormal parametrisation.
    Stress months: AUM-weighted stress rates (deterministic override).

    Parameters
    ----------
    reg      : dict, entry from INVESTOR_REGISTRY
    n_months : int, number of periods to simulate (default 12)

    Returns list of n_months floats (fraction of total NAV per period).
    """
    rng    = np.random.default_rng(reg['seed'])
    sigma  = reg['sigma']
    stress = reg['stress_months']
    sched  = []
    for m in range(1, n_months + 1):
        rate = 0.0
        for _, w, normal_r, stress_r in reg['investors']:
            if m in stress:
                rate += w * stress_r
            else:
                # lognormal with mean = normal_r: mu = log(normal_r) - sigma^2/2
                mu    = np.log(normal_r) - 0.5 * sigma ** 2
                rate += w * float(rng.lognormal(mean=mu, sigma=sigma))
        sched.append(float(np.clip(rate, 0.0, 1.0)))
    return sched


# ---- Visualise generated schedules -----------------------------------------
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (fid, reg) in zip(axes, INVESTOR_REGISTRY.items()):
    sched  = build_redemption_schedule(reg)
    months = list(range(1, 13))
    colors = [ACCENT2 if m in reg['stress_months'] else ACCENT for m in months]

    ax.bar(months, [r * 100 for r in sched], color=colors, width=0.6)

    w_norm   = sum(w * nr for _, w, nr, _  in reg['investors'])
    w_stress = sum(w * sr for _, w, _,  sr in reg['investors'])
    ax.axhline(w_norm   * 100, color='white',  linewidth=1.2, linestyle='--',
               label=f'Wtd normal rate ({w_norm*100:.1f}% pm)')
    ax.axhline(w_stress * 100, color=ACCENT3,  linewidth=1.2, linestyle='--',
               label=f'Wtd stress rate ({w_stress*100:.1f}% pm)')

    handles, labels = ax.get_legend_handles_labels()
    handles = [Patch(facecolor=C['blue2'],  label='Normal month'),
               Patch(facecolor=C['amber'], label='Stress month')] + handles
    ax.legend(handles=handles, fontsize=7)
    ax.set_xticks(months)
    ax.set_xlabel('Month')
    ax.set_ylabel('% of NAV')
    ax.set_title(reg['label'])

fig.suptitle('Generated Redemption Schedules -- Investor Base Model',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ---- Investor composition tables ------------------------------------------
for fid, reg in INVESTOR_REGISTRY.items():
    print(f"\n{reg['label']}  |  sigma={reg['sigma']}  |  stress months: {sorted(reg['stress_months'])}")
    rows = [
        {
            'Investor type'   : t,
            'AUM share'       : f'{w:.0%}',
            'Normal rate (pm)': f'{nr:.1%}',
            'Stress rate (pm)': f'{sr:.1%}',
            'Wtd normal'      : f'{w*nr:.2%}',
            'Wtd stress'      : f'{w*sr:.2%}',
        }
        for t, w, nr, sr in reg['investors']
    ]
    display(pd.DataFrame(rows).set_index('Investor type'))

### 3. LMT Trigger Analysis
**AIFMD II (MRS-84)**

The simulation applies to **open-ended funds with a liquid portfolio only**: UCITS Balanced and AIFM Hedge Fund.  
AIFM Private Debt and AIFM Real Estate hold predominantly illiquid assets (loans and direct property). For those funds any redemption wave rapidly exhausts the liquid buffer in the first one or two gated periods; the analytically interesting question there is *how many months until the liquid sleeve hits zero*, not the gate/swing dynamics — see Section 3.  
PE Buyout and Infra Core are closed-ended: no redemption obligation, no LMT tools required.

---

#### Regulatory basis

- **AIFMD II** — Directive 2024/927/EU, Art. 16a: open-ended AIFs must activate at least two LMTs in stress
- **ESMA Guidelines** — ESMA34-671404336-1364 (April 2025): defines activation conditions, board governance requirements, and reporting obligations
- **Del. Reg. 231/2013** — Articles 46–49: liquidity management framework; Art. 95: LMT policy content

---

#### Three LMTs modelled

| Tool | Type | Trigger condition | Action |
|---|---|---|---|
| Redemption gate | Quantitative | Gross demand $> G \cdot V_t$ | Cap paid at $\min(G \cdot V_t,\; L_t)$; defer excess |
| Swing pricing | Anti-dilution | Gross rate $> S$ | Adjust NAV per unit by levy $\delta$ |
| Suspension | Exceptional | Board decision (simulated: two conditions) | Zero redemptions paid |

where $G$ = gate threshold, $V_t$ = total NAV at $t$, $L_t$ = liquid sleeve at $t$, $S$ = swing threshold, $\delta$ = swing factor.

---

#### Model mechanics

##### 1. Contagion feedback

When the gate fires in period $t-1$, rational remaining investors accelerate their own redemption requests to avoid being queue-prioritised behind a growing backlog. This is modelled by scaling the base schedule:

$$r_t^{\text{eff}} = \begin{cases} \min\!\left(r_t^{\text{base}} \cdot \lambda,\; 1\right) & \text{if gate was active in } t-1 \\ r_t^{\text{base}} & \text{otherwise} \end{cases}$$

where $\lambda \geq 1$ is the **contagion multiplier**. Calibration: $\lambda = 1.3$ for UCITS (mixed retail/institutional); $\lambda = 1.5$ for hedge funds (concentrated institutional base with faster reaction time).

The effective gross demand in EUR is then:

$$D_t^{\text{new}} = r_t^{\text{eff}} \cdot V_t$$

##### 2. Gate mechanics

Total demand includes new requests and the outstanding backlog $B_{t-1}$:

$$D_t^{\text{total}} = D_t^{\text{new}} + B_{t-1}$$

The **gate cap** is the lower of the contractual limit (a fraction of total NAV) and what the fund can actually liquidate:

$$C_t = \min\!\left(G \cdot V_t,\; L_t\right)$$

The gate fires when $D_t^{\text{total}} > C_t$. When active:

$$\text{Paid}_t = \min(C_t,\; D_t^{\text{total}})$$

$$\text{Deferred}_t = \max\!\left(0,\; D_t^{\text{new}} - \max(0,\; C_t - B_{t-1})\right)$$

$$B_t = \max\!\left(0,\; D_t^{\text{total}} - \text{Paid}_t\right)$$

The liquid sleeve evolves as:

$$L_t = L_{t-1} - \text{Paid}_t$$

##### 3. Swing pricing

The swing activates when $r_t^{\text{eff}} > S$. A dilution levy $\delta$ is applied to the effective NAV per unit, adjusting the redemption price upward so that transaction costs are borne by redeeming investors rather than the remaining fund:

$$\text{NAV}_t^{\text{eff}} = V_t \cdot (1 + \delta)$$

This does not reduce the fund's cash outflow — it increases the effective redemption price, protecting remaining investors from dilution.

##### 4. Suspension (simulation approximation)

Suspension triggers automatically in the model when two conditions hold simultaneously:

$$\text{consec\_gate} \geq N_{\text{suspend}} \quad \text{AND} \quad \frac{B_t}{L_t} \geq \theta_B$$

where $N_{\text{suspend}}$ is the minimum consecutive gate months and $\theta_B$ is the backlog-to-liquid-NAV threshold.

**Important**: under AIFMD II and ESMA Guidelines (ESMA34-671404336-1364), actual suspension requires exceptional circumstances and a formal board decision — it cannot be automatic. The auto-trigger is a simulation convenience for stress-testing only.

#### 3.1 UCITS Balanced

**Portfolio**: multi-asset (equities, IG/HY bonds, listed alternatives). Liquid sleeve estimated at **85% of NAV** — the remaining 15% is allocated to less-liquid instruments (HY bonds and listed alternatives with lower daily turnover).

**Parameters**:

| Parameter | Value | Rationale |
|---|---|---|
| Gate threshold $G$ | 10% of NAV | Standard UCITS gate; contractual fund document limit |
| Swing threshold $S$ | 5% of NAV | Activates when net flows are large enough to impose material transaction costs |
| Swing factor $\delta$ | 50 bps | Market impact estimate for the liquid sleeve |
| Contagion multiplier $\lambda$ | 1.3 | Mixed retail/institutional base; slower reaction than pure HF |
| Suspension conditions | $N = 3$ consecutive gate months; $\theta_B = 25\%$ | Conservative — UCITS retail investor protections favour later, deliberate board decision |

**Stress schedule**: two consecutive spike months (months 3–4 at 10–14% of NAV) followed by sustained moderate outflow. Total 12-month gross request ≈ 61% of initial NAV.

In [ ]:
LMT_PARAMS = {
    'UCITS_Balanced': {
        'label'          : INVESTOR_REGISTRY['UCITS_Balanced']['label'],
        'liquid_pct'     : 0.85,
        'gate_threshold' : 0.10,
        'swing_threshold': 0.05,
        'contagion'      : 1.3,
        'consec_gate'    : 3,
        'backlog_pct'    : 0.25,
        'schedule'       : build_redemption_schedule(INVESTOR_REGISTRY['UCITS_Balanced']),
    },
    'AIFM_HedgeFund': {
        'label'          : INVESTOR_REGISTRY['AIFM_HedgeFund']['label'],
        'liquid_pct'     : 0.75,
        'gate_threshold' : 0.10,
        'swing_threshold': 0.05,
        'contagion'      : 1.5,
        'consec_gate'    : 3,
        'backlog_pct'    : 0.25,
        'schedule'       : build_redemption_schedule(INVESTOR_REGISTRY['AIFM_HedgeFund']),
    },
}


def plot_lmt(df, label):
    months = df['month'].values
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.bar(months, df['paid_eur']     / 1e6, color=ACCENT,  label='Paid',     width=0.5)
    ax.bar(months, df['deferred_eur'] / 1e6, color=ACCENT2, label='Deferred',
           bottom=df['paid_eur'] / 1e6, width=0.5)
    ax2 = ax.twinx()
    ax2.plot(months, df['backlog_eur'] / 1e6, color=ACCENT3,
             marker='o', linewidth=1.5, label='Backlog (rhs)')
    ax2.set_ylabel('Backlog (EUR m)', fontsize=8)
    ax.set_xlabel('Month'); ax.set_ylabel('EUR m')
    ax.set_title('Paid / Deferred / Backlog')
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)

    ax = axes[1]
    ax.stackplot(months,
                 df['liquid_nav_eur']   / 1e6,
                 df['illiquid_nav_eur'] / 1e6,
                 labels=['Liquid NAV', 'Illiquid NAV'],
                 colors=[ACCENT, ACCENT2], alpha=0.8)
    ax.set_xlabel('Month'); ax.set_ylabel('EUR m')
    ax.set_title('NAV Evolution')
    ax.legend(loc='upper right', fontsize=7)

    ax = axes[2]
    for i, (col, color, lbl) in enumerate([
        ('gate_active',       ACCENT,  'Gate'),
        ('swing_active',      ACCENT2, 'Swing'),
        ('suspension_active', ACCENT3, 'Suspension'),
    ]):
        ax.step(months, df[col].astype(int) + i * 1.2,
                where='post', color=color, linewidth=2, label=lbl)
    ax.set_yticks([0, 1.2, 2.4])
    ax.set_yticklabels(['Gate', 'Swing', 'Suspension'], fontsize=8)
    ax.set_xlabel('Month'); ax.set_title('LMT Flags Active')
    ax.set_ylim(-0.3, 3.5)

    fig.suptitle(f'LMT Trigger Analysis -- {label}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
p   = LMT_PARAMS['UCITS_Balanced']
pos = pd.read_sql(
    "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
    ENGINE, params={'fid': 'UCITS_Balanced', 'dt': VALUATION_DATE}
)
nav = pos['market_value_eur'].sum()

df_ucits = lmt_trigger_analysis(
    nav                             = nav,
    liquid_pct                      = p['liquid_pct'],
    gate_threshold                  = p['gate_threshold'],
    swing_threshold                 = p['swing_threshold'],
    redemption_schedule             = p['schedule'],
    consecutive_gate_for_suspension = p['consec_gate'],
    backlog_pct_for_suspension      = p['backlog_pct'],
    contagion_multiplier            = p['contagion'],
)

display(df_ucits.style.format({
    'base_gross_pct'      : '{:.2f}%',
    'effective_gross_pct' : '{:.2f}%',
    'effective_gross_eur' : '€{:,.0f}',
    'paid_eur'            : '€{:,.0f}',
    'deferred_eur'        : '€{:,.0f}',
    'backlog_eur'         : '€{:,.0f}',
    'liquid_nav_eur'      : '€{:,.0f}',
    'illiquid_nav_eur'    : '€{:,.0f}',
    'total_nav_eur'       : '€{:,.0f}',
}))

plot_lmt(df_ucits, p['label'])

#### 3.2 AIFM Hedge Fund

**Portfolio**: long/short equity with derivatives overlay. Liquid sleeve estimated at **75% of NAV** — the remaining 25% consists of less-liquid mid/small-cap positions, structured products, and OTC derivatives with longer unwind horizons.

**Parameters**:

| Parameter | Value | Rationale |
|---|---|---|
| Gate threshold $G$ | 10% of NAV | Standard AIFMD gate; fund document limit |
| Swing threshold $S$ | 5% of NAV | Same level as UCITS; fund is similarly liquid on the asset side |
| Swing factor $\delta$ | 50 bps | Market impact estimate |
| Contagion multiplier $\lambda$ | 1.5 | Concentrated institutional investor base (pension funds, family offices, funds-of-funds); faster and larger reaction when gate fires |
| Suspension conditions | $N = 3$ consecutive gate months; $\theta_B = 25\%$ | Same threshold as UCITS; board governance applies equally |

**Stress schedule**: slightly more severe than UCITS — month 4 peaks at 15% gross — reflecting the tendency of institutional HF investors to concentrate redemptions during market dislocations (liquidity crisis, manager performance concerns). Total 12-month gross request ≈ 70% of initial NAV before contagion scaling.

**Key difference from UCITS**: the higher $\lambda = 1.5$ means a gate in month $t$ can amplify the month $t+1$ request by 50%, making early gate activation a self-reinforcing dynamic. This is the primary reason HF liquidity stress is structurally more severe than UCITS despite a similar asset-side profile.

In [ ]:
p   = LMT_PARAMS['AIFM_HedgeFund']
pos = pd.read_sql(
    "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
    ENGINE, params={'fid': 'AIFM_HedgeFund', 'dt': VALUATION_DATE}
)
nav = pos['market_value_eur'].sum()

df_hf = lmt_trigger_analysis(
    nav                             = nav,
    liquid_pct                      = p['liquid_pct'],
    gate_threshold                  = p['gate_threshold'],
    swing_threshold                 = p['swing_threshold'],
    redemption_schedule             = p['schedule'],
    consecutive_gate_for_suspension = p['consec_gate'],
    backlog_pct_for_suspension      = p['backlog_pct'],
    contagion_multiplier            = p['contagion'],
)

display(df_hf.style.format({
    'base_gross_pct'      : '{:.2f}%',
    'effective_gross_pct' : '{:.2f}%',
    'effective_gross_eur' : '€{:,.0f}',
    'paid_eur'            : '€{:,.0f}',
    'deferred_eur'        : '€{:,.0f}',
    'backlog_eur'         : '€{:,.0f}',
    'liquid_nav_eur'      : '€{:,.0f}',
    'illiquid_nav_eur'    : '€{:,.0f}',
    'total_nav_eur'       : '€{:,.0f}',
}))

plot_lmt(df_hf, p['label'])

### 4. Cross-Fund Summary


In [ ]:

results = {
    'UCITS Balanced' : df_ucits,
    'AIFM Hedge Fund': df_hf,
}

summary_rows = []
for fname, df in results.items():
    gate_m = df[df['gate_active']]['month'].tolist()
    swing_m = df[df['swing_active']]['month'].tolist()
    susp_m  = df[df['suspension_active']]['month'].tolist()
    summary_rows.append({
        'Fund'                       : fname,
        'Gate first fires (month)'   : gate_m[0]  if gate_m  else '—',
        'Swing first fires (month)'  : swing_m[0] if swing_m else '—',
        'Suspension triggers (month)': susp_m[0]  if susp_m  else '—',
        'Peak backlog (€m)'          : round(df['backlog_eur'].max() / 1e6, 1),
        'Total paid (€m)'            : round(df['paid_eur'].sum() / 1e6, 1),
        'Terminal liquid NAV (€m)'   : round(df['liquid_nav_eur'].iloc[-1] / 1e6, 1),
    })

pd.DataFrame(summary_rows).set_index('Fund')

## Illiquids

### 1. Liquid Buffer Exhaustion

For **Private Debt** and **Real Estate**, the liquid sleeve represents only **15–20% of total NAV**. The gate cap under any realistic threshold (10–15% of NAV) exceeds the liquid sleeve within the first one or two periods of any moderate stress scenario. Once the liquid buffer is exhausted the fund literally cannot pay redemptions regardless of the contractual gate cap — the gate becomes irrelevant and suspension is the only remaining tool.

The analytically useful question for these funds is therefore not *when does the gate fire* but rather:

> **How many months does the liquid buffer last under different constant redemption outflow rates, with and without a gate?**

#### Model

Let $L_0 = \alpha \cdot V_0$ be the initial liquid sleeve, where $\alpha \in \{0.15, 0.20\}$ for RE and PD respectively.

Under a **constant monthly outflow rate** $r$ (as a fraction of *initial* NAV $V_0$, assumed fixed for simplicity):

$$L_t = L_0 - r \cdot V_0 \cdot t \quad \text{(no gate)}$$

The fund exhausts its liquid buffer at:

$$t^* = \left\lfloor \frac{L_0}{r \cdot V_0} \right\rfloor = \left\lfloor \frac{\alpha}{r} \right\rfloor \text{ months}$$

With a **gate** capping monthly outflows at $G \cdot V_0$:

$$L_t = L_0 - \min(r, G) \cdot V_0 \cdot t$$

$$t^*_{\text{gate}} = \left\lfloor \frac{\alpha}{\min(r, G)} \right\rfloor \text{ months}$$

When $r \leq G$ the gate does not bind and $t^* = t^*_{\text{gate}}$. When $r > G$ the gate extends survival by a factor of $r / G$ relative to the ungated case. Note that deferred redemptions accumulate as a backlog — the gate buys time but does not reduce total outflow obligation.

In [ ]:
ILLIQUID_FUNDS = {
    'AIFM_PrivateDebt': {'label': 'AIFM Private Debt', 'liquid_pct': 0.20, 'gate': 0.10},
    'AIFM_RealEstate':  {'label': 'AIFM Real Estate',  'liquid_pct': 0.15, 'gate': 0.10},
}

OUTFLOW_RATES = [0.03, 0.05, 0.08, 0.10, 0.15, 0.20]  # monthly as fraction of initial NAV
MONTHS        = list(range(1, 25))                       # 24-month horizon

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (fid, cfg) in zip(axes, ILLIQUID_FUNDS.items()):
    pos = pd.read_sql(
        "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
        ENGINE, params={'fid': fid, 'dt': VALUATION_DATE}
    )
    nav      = pos['market_value_eur'].sum()
    liquid_0 = nav * cfg['liquid_pct']
    gate_cap = cfg['gate'] * nav   # monthly gate cap in EUR (fixed to initial NAV)

    for rate in OUTFLOW_RATES:
        monthly_outflow = rate * nav
        effective_out   = min(monthly_outflow, gate_cap)   # gated outflow

        # months until liquid sleeve hits zero (gated)
        if effective_out > 0:
            exhaustion = liquid_0 / effective_out
        else:
            exhaustion = float('inf')

        liquid_path = [max(0.0, liquid_0 - effective_out * m) / 1e6 for m in MONTHS]
        label = f'{rate*100:.0f}% p.m.'
        ax.plot(MONTHS, liquid_path, marker='o', markersize=3, linewidth=1.5,
                label=label)
        if exhaustion <= 24:
            ax.axvline(exhaustion, color='grey', linewidth=0.5, alpha=0.4)

    ax.axhline(0, color='red', linewidth=1, linestyle='--', label='Buffer exhausted')
    ax.set_title(f"{cfg['label']}\n(liquid sleeve = {cfg['liquid_pct']*100:.0f}% NAV, "
                 f"gate = {cfg['gate']*100:.0f}% NAV/month)", fontsize=10)
    ax.set_xlabel('Month')
    ax.set_ylabel('Liquid sleeve (€m)')
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle('Liquid Buffer Exhaustion Under Constant Gated Outflows — Illiquid Funds',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary table: months to exhaustion ──────────────────────────────────────
print("\nMonths to liquid buffer exhaustion (gated at 10% of initial NAV/month):\n")
rows = []
for fid, cfg in ILLIQUID_FUNDS.items():
    pos      = pd.read_sql(
        "SELECT * FROM positions WHERE fund_id = :fid AND date = :dt",
        ENGINE, params={'fid': fid, 'dt': VALUATION_DATE}
    )
    nav      = pos['market_value_eur'].sum()
    liquid_0 = nav * cfg['liquid_pct']
    gate_cap = cfg['gate'] * nav
    r = {'Fund': cfg['label']}
    for rate in OUTFLOW_RATES:
        eff = min(rate * nav, gate_cap)
        t   = int(liquid_0 // eff) if eff > 0 else 999
        r[f'{rate*100:.0f}% p.m.'] = t if t <= 24 else '>24'
    rows.append(r)
display(pd.DataFrame(rows).set_index('Fund'))

### 2. PE Buyout and Infra Core

Both funds are **closed-ended** with no redemption obligation to investors during the fund's life. Capital is called and returned on the fund's own schedule via capital calls and distributions. AIFMD II LMT tools (gates, swing pricing, suspension) do not apply.

Liquidity management for these funds is addressed through separate analytical frameworks:

| Framework | Tool | Location |
|---|---|---|
| Unfunded commitment monitoring | Capital call facility sizing | `aifm_pe_buyout.ipynb`, `aifm_infra_ fund.ipynb` |
| Asset-level debt headroom | DSCR / LTV covenant tracking | `infra_utils.py` — `dscr_profile`, `ltv_profile` |
| Controlled capital return | Distribution waterfall, DPI/RVPI | `pe_utils.py` — `pe_multiples`, `fund_irr` |